# Energy Project Queue Analysis

Author: Sophie McDowall
DSAN 5550 

# Data Loading

I load data from the LBNL queue.

In [54]:
# imports and start for carbon tracking
from datetime import datetime 
from codecarbon import EmissionsTracker
import os 

# start carbon tracking
tracker = EmissionsTracker(
    project_name="transmission-queue-model",
    output_file="../outputs/codecarbon_emissions.csv",
    log_level="error"
)
tracker.start()
print("CodeCarbon tracking started.")


CodeCarbon tracking started.


In [55]:
# load lbnl queue df

import pandas as pd
import numpy as np

# read in data
df = pd.read_csv("../data/LBNL_queue_only.csv")

# Display the first 5 rows
print(df.shape)
print(df.columns)


(36441, 31)
Index(['q_id', 'q_status', 'q_date', 'prop_date', 'on_date', 'wd_date',
       'ia_date', 'IA_status_raw', 'IA_status_clean', 'county', 'state',
       'county_state_pairs', 'fips_codes', 'poi_name', 'region',
       'project_name', 'utility', 'entity', 'developer', 'cluster', 'service',
       'project_type', 'type1', 'type2', 'type3', 'mw1', 'mw2', 'mw3',
       'type_clean', 'q_year', 'prop_year'],
      dtype='object')


In [39]:
# date columns are in excel format, switch to normal
date_cols = ["q_date", "prop_date", "on_date", "wd_date", "ia_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], unit='D', origin='1899-12-30', errors='coerce')

# print dates to verify
for col in date_cols:
    print(col, df[col].dtype, '|', df[col].min(), '-', df[col].max())

q_date datetime64[ns] | 1970-01-01 00:00:00 - 2024-12-31 00:00:00
prop_date datetime64[ns] | 1905-06-23 00:00:00 - 2050-12-31 00:00:00
on_date datetime64[ns] | 1924-02-28 00:00:00 - 2024-12-30 00:00:00
wd_date datetime64[ns] | 1999-06-26 00:00:00 - 2025-02-19 00:00:00
ia_date datetime64[ns] | 1970-01-01 00:00:00 - 2024-12-30 00:00:00


In [40]:
# set floors for data based on known values
df.loc[df['q_date'] < '2000-01-01', 'q_date'] = pd.NaT

df.loc[df['prop_date'] < '1990-01-01', 'prop_date'] = pd.NaT

df.loc[df['on_date'] < '2000-01-01', 'on_date'] = pd.NaT

df.loc[df['ia_date'] < '1990-01-01', 'ia_date'] = pd.NaT

df.loc[df['wd_date'] < '1999-01-01', 'wd_date'] = pd.NaT

# view final null counts and date ranges
for col in ['q_date', 'prop_date', 'on_date', 'wd_date', 'ia_date']:
    print(col, '|', df[col].min(), '-', df[col].max(), '| nulls:', df[col].isna().sum())


q_date | 2000-01-01 00:00:00 - 2024-12-31 00:00:00 | nulls: 1069
prop_date | 1990-03-01 00:00:00 - 2050-12-31 00:00:00 | nulls: 7421
on_date | 2000-01-08 00:00:00 - 2024-12-30 00:00:00 | nulls: 33543
wd_date | 1999-06-26 00:00:00 - 2025-02-19 00:00:00 | nulls: 26303
ia_date | 1999-08-25 00:00:00 - 2024-12-30 00:00:00 | nulls: 34057


In [42]:
pd.DataFrame({
    'count': df.count(),
    'nulls': df.isna().sum(),
    'null_pct': (df.isna().mean() * 100).round(1)
}).sort_values('nulls', ascending=False)

,count,nulls,null_pct
mw3,45,36396,99.9
type3,99,36342,99.7
mw2,1156,35285,96.8
ia_date,2384,34057,93.5
on_date,2898,33543,92.0
type2,4233,32208,88.4
developer,6729,29712,81.5
cluster,9505,26936,73.9
wd_date,10138,26303,72.2
project_name,12184,24257,66.6


In [44]:
# projects with IA status but no date
mask = df['IA_status_clean'].notna() & df['ia_date'].isna()
print(f"Has IA status but no ia_date: {mask.sum():,}")
print(df.loc[mask, 'IA_status_clean'].value_counts())

Has IA status but no ia_date: 28,604
IA_status_clean
Withdrawn                      6598
In Progress (unknown study)    5074
System Impact Study            4997
IA Executed                    3217
Facility Study                 3108
Feasibility Study              2062
Cluster Study                  1542
Operational                    1220
Suspended                       332
Not Started                     276
IA Pending                      132
Construction                     45
Combined                          1
Name: count, dtype: int64


In [46]:
# the most important check — what does your actual outcome distribution look like
print(df['q_status'].value_counts(normalize=True).round(3) * 100)

q_status
withdrawn      57.4
active         28.9
operational    12.2
suspended       1.5
unknown         0.0
Name: proportion, dtype: float64


In [51]:
# ═══════════════════════════════════════════════════════════════════════════════
# INITIAL DATA INSPECTION
# ═══════════════════════════════════════════════════════════════════════════════

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nNull counts:\n", df.isna().sum().sort_values(ascending=False))
print("\nSample rows:")
print(df.head(3))

# ── check q_status values before anything else ────────────────────────────────
print("\nq_status values:")
print(df['q_status'].value_counts(dropna=False))

# ── check type columns to inform is_clean derivation ─────────────────────────
for col in ['type1', 'type2', 'type3', 'type_clean']:
    if col in df.columns:
        print(f"\n{col} top values:")
        print(df[col].value_counts(dropna=False).head(20))

Shape: (36441, 31)

Columns: ['q_id', 'q_status', 'q_date', 'prop_date', 'on_date', 'wd_date', 'ia_date', 'IA_status_raw', 'IA_status_clean', 'county', 'state', 'county_state_pairs', 'fips_codes', 'poi_name', 'region', 'project_name', 'utility', 'entity', 'developer', 'cluster', 'service', 'project_type', 'type1', 'type2', 'type3', 'mw1', 'mw2', 'mw3', 'type_clean', 'q_year', 'prop_year']

Dtypes:
 q_id                   object
q_status               object
q_date                float64
prop_date             float64
on_date               float64
wd_date               float64
ia_date               float64
IA_status_raw          object
IA_status_clean        object
county                 object
state                  object
county_state_pairs     object
fips_codes            float64
poi_name               object
region                 object
project_name           object
utility                object
entity                 object
developer              object
cluster                objec

In [53]:
import pandas as pd
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 0 — INITIAL CLEANUP
# Drop the one row with no q_id and rows with no q_date (no temporal anchor).
# Drop unknown status rows (only 4) — unclassifiable outcome.
# Reset index after drops so all downstream operations are clean.
# ═══════════════════════════════════════════════════════════════════════════════

resolved = df.copy()
resolved = resolved.dropna(subset=['q_id'])
resolved = resolved[resolved['q_status'] != 'unknown'].reset_index(drop=True)

print(f"Rows after initial drop: {len(resolved):,}")


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — DATE PARSING
# All date columns are stored as Excel serial numbers (float).
# Convert using origin='1899-12-30' (Excel epoch).
# Derive q_year, prop_year, q_quarter from parsed dates — more reliable
# than the raw float columns.
# ═══════════════════════════════════════════════════════════════════════════════

date_cols = ['q_date', 'prop_date', 'on_date', 'wd_date', 'ia_date']
for col in date_cols:
    resolved[col] = pd.to_datetime(
        resolved[col], unit='D', origin='1899-12-30', errors='coerce'
    )

resolved['q_year']    = resolved['q_date'].dt.year
resolved['prop_year'] = resolved['prop_date'].dt.year
resolved['q_quarter'] = resolved['q_date'].dt.quarter

# drop rows with no q_date — no temporal anchor, unusable
resolved = resolved.dropna(subset=['q_date']).reset_index(drop=True)

print(f"Rows after dropping null q_date: {len(resolved):,}")
print(f"q_year range: {resolved['q_year'].min():.0f} – {resolved['q_year'].max():.0f}")


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — OUTCOME VARIABLE
# withdrawn = 1 if q_status is 'withdrawn', 0 otherwise.
# Suspended projects are kept in the dataset but excluded from the
# classifier (ambiguous outcome). Flagged here for easy filtering later.
# ═══════════════════════════════════════════════════════════════════════════════

resolved['withdrawn']  = (resolved['q_status'] == 'withdrawn').astype(int)
resolved['suspended']  = (resolved['q_status'] == 'suspended').astype(int)

print(f"\nOutcome distribution:")
print(resolved['q_status'].value_counts())
print(f"\nWithdrawn: {resolved['withdrawn'].sum():,}  ({resolved['withdrawn'].mean():.1%})")


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — TIME IN QUEUE
# For withdrawn:   wd_date - q_date
# For operational: on_date - q_date
# For active/suspended (censored): CUTOFF_DATE - q_date
#
# Withdrawn projects missing wd_date (common in FERC data — withdrawal
# recorded in status but date not captured) are imputed via progressive
# cohort medians: region+year → region → year → global median.
# Imputed rows are flagged.
# ═══════════════════════════════════════════════════════════════════════════════

CUTOFF_DATE = pd.Timestamp('2024-12-31')

resolved['days_in_queue'] = np.where(
    resolved['withdrawn'] == 1,
    (resolved['wd_date'] - resolved['q_date']).dt.days,
    np.where(
        resolved['on_date'].notna(),
        (resolved['on_date'] - resolved['q_date']).dt.days,
        (CUTOFF_DATE - resolved['q_date']).dt.days
    )
).clip(min=0)

needs_imputation = (resolved['withdrawn'] == 1) & (resolved['days_in_queue'].isna())
print(f"\nWithdrawn rows needing duration imputation: {needs_imputation.sum():,}")

# build cohort medians from observed withdrawn rows only
observed_wd = resolved[
    (resolved['withdrawn'] == 1) & (resolved['days_in_queue'].notna())
]
l1 = (observed_wd.groupby(['region', 'q_year'])['days_in_queue']
      .median().reset_index().rename(columns={'days_in_queue': 'l1'}))
l2 = (observed_wd.groupby('region')['days_in_queue']
      .median().reset_index().rename(columns={'days_in_queue': 'l2'}))
l3 = (observed_wd.groupby('q_year')['days_in_queue']
      .median().reset_index().rename(columns={'days_in_queue': 'l3'}))
l4 = observed_wd['days_in_queue'].median()

resolved = (resolved
    .merge(l1, on=['region', 'q_year'], how='left')
    .merge(l2, on='region',             how='left')
    .merge(l3, on='q_year',             how='left')
).reset_index(drop=True)

needs_imputation = (resolved['withdrawn'] == 1) & (resolved['days_in_queue'].isna())

resolved.loc[needs_imputation, 'days_in_queue'] = (
    resolved.loc[needs_imputation, 'l1']
    .fillna(resolved.loc[needs_imputation, 'l2'])
    .fillna(resolved.loc[needs_imputation, 'l3'])
    .fillna(l4)
)
resolved = resolved.drop(columns=['l1', 'l2', 'l3'])
resolved['days_in_queue_imputed'] = needs_imputation.astype(int)

print(f"Remaining nulls in days_in_queue: {resolved['days_in_queue'].isna().sum()}")


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — INTERCONNECTION AGREEMENT STAGE
# IA_status_clean is the reliable column (ia_date is 93% null).
# Encoded as ordinal 0–6: higher = further through the study process.
# has_ia flags projects that reached a signed or executed agreement.
# ═══════════════════════════════════════════════════════════════════════════════

ia_stage_order = {
    'Not Started':                 0,
    'Withdrawn':                   0,
    'In Progress (unknown study)': 1,
    'Suspended':                   1,
    'Feasibility Study':           2,
    'Cluster Study':               2,
    'Combined':                    3,
    'System Impact Study':         3,
    'Facility Study':              4,
    'IA Pending':                  5,
    'IA Executed':                 6,
    'Construction':                6,
    'Operational':                 6,
}
resolved['ia_stage'] = (resolved['IA_status_clean']
                         .map(ia_stage_order)
                         .fillna(0)
                         .astype(int))
resolved['has_ia'] = (resolved['ia_stage'] >= 5).astype(int)

print(f"\nIA stage distribution:")
print(resolved['ia_stage'].value_counts().sort_index())


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — PROJECT SIZE AND TECHNOLOGY
# mw_total sums capacity across all technology components.
# is_collocated: project has a secondary technology (type2 present).
# has_storage: any storage component across type1/type2/type3.
# is_clean: primary technology is renewable/zero-carbon.
# is_fossil: primary technology is fossil fuel.
# ═══════════════════════════════════════════════════════════════════════════════

resolved['mw1'] = pd.to_numeric(resolved['mw1'], errors='coerce')
resolved['mw2'] = pd.to_numeric(resolved['mw2'], errors='coerce')
resolved['mw3'] = pd.to_numeric(resolved['mw3'], errors='coerce')
resolved['mw1'] = resolved['mw1'].fillna(resolved['mw1'].median())

resolved['mw_total'] = (resolved[['mw1', 'mw2', 'mw3']]
                         .sum(axis=1, min_count=1)
                         .fillna(resolved['mw1']))

resolved['is_collocated'] = resolved['type2'].notna().astype(int)

storage_terms = ['Battery', 'Storage', 'BESS']
resolved['has_storage'] = (
    resolved[['type1', 'type2', 'type3']]
    .apply(lambda col: col.str.contains('|'.join(storage_terms), na=False))
    .any(axis=1).astype(int)
)

clean_types  = ['Solar', 'Wind', 'Battery', 'Storage', 'Hydro',
                 'Geothermal', 'Offshore Wind', 'Biofuel',
                 'Pumped Storage', 'Hydrogen', 'Flywheel', 'Wave']
fossil_types = ['Gas', 'Coal', 'Oil', 'Diesel',
                 'Nuclear', 'Combustion Turbine', 'Combined Cycle']

resolved['is_clean'] = (
    resolved['type_clean'].str.contains(
        '|'.join(clean_types), case=False, na=False
    )
).astype(int)

resolved['is_fossil'] = (
    resolved['type_clean'].str.contains(
        '|'.join(fossil_types), case=False, na=False
    )
).astype(int)

print(f"\nTechnology flags:")
print(f"  Collocated:  {resolved['is_collocated'].sum():,}  ({resolved['is_collocated'].mean():.1%})")
print(f"  Has storage: {resolved['has_storage'].sum():,}  ({resolved['has_storage'].mean():.1%})")
print(f"  Clean:       {resolved['is_clean'].sum():,}  ({resolved['is_clean'].mean():.1%})")
print(f"  Fossil:      {resolved['is_fossil'].sum():,}  ({resolved['is_fossil'].mean():.1%})")


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — PROPOSAL LEAD TIME
# Days between queue entry and proposed online date.
# Longer lead = larger or more complex project.
# Missing prop_date is flagged separately — may signal speculative filing.
# ═══════════════════════════════════════════════════════════════════════════════

resolved['prop_lead_days']    = (
    (resolved['prop_date'] - resolved['q_date']).dt.days.clip(lower=0)
)
resolved['prop_date_missing'] = resolved['prop_date'].isna().astype(int)
resolved['prop_lead_days']    = resolved['prop_lead_days'].fillna(
    resolved['prop_lead_days'].median()
)


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — QUEUE CONGESTION (REGION LEVEL)
# region_year_congestion: number of projects entering same region+year.
# congestion_yoy: year-over-year growth rate of queue entries in that region.
# Captures the post-2019 solar+storage surge dynamic.
# ═══════════════════════════════════════════════════════════════════════════════

resolved['region_year_congestion'] = (
    resolved.groupby(['region', 'q_year'])['q_id'].transform('count')
)

region_year_counts = (resolved.groupby(['region', 'q_year'])
                               .size().reset_index(name='n'))
region_year_counts['congestion_yoy'] = (
    region_year_counts.groupby('region')['n']
    .pct_change().fillna(0).clip(-1, 5)
)
resolved = resolved.merge(
    region_year_counts[['region', 'q_year', 'congestion_yoy']],
    on=['region', 'q_year'], how='left'
)


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 8 — POI CROWDING (NODE LEVEL)
# More granular than region-level — counts projects competing for the
# same substation or interconnection point in the same year.
# Missing POI flagged separately.
# ═══════════════════════════════════════════════════════════════════════════════

resolved['poi_total_projects'] = (
    resolved.groupby('poi_name')['q_id'].transform('count')
)
resolved['poi_year_projects']  = (
    resolved.groupby(['poi_name', 'q_year'])['q_id'].transform('count')
)
resolved['poi_missing']        = resolved['poi_name'].isna().astype(int)
resolved['poi_total_projects'] = resolved['poi_total_projects'].fillna(0)
resolved['poi_year_projects']  = resolved['poi_year_projects'].fillna(0)


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 9 — CLUSTER MEMBERSHIP
# Projects in a cluster study share interconnection costs with neighbors.
# If cluster neighbors withdraw, remaining projects face higher costs.
# cluster_size captures cascade risk — larger clusters = more exposure.
# ═══════════════════════════════════════════════════════════════════════════════

resolved['in_cluster']   = resolved['cluster'].notna().astype(int)
resolved['cluster_size'] = (
    resolved[resolved['cluster'].notna()]
    .groupby('cluster')['q_id'].transform('count')
)
resolved['cluster_size'] = resolved['cluster_size'].fillna(0)


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 10 — DEVELOPER EXPERIENCE
# Prior project count for the same developer, ordered by queue entry date.
# Captures speculative filing behavior — first-time filers withdraw more.
# Null developers get a unique sentinel so they are not grouped together
# (each unknown developer is treated as a distinct one-off filer).
# ═══════════════════════════════════════════════════════════════════════════════

resolved['developer_clean'] = resolved['developer'].fillna(
    'unknown_' + resolved['q_id'].astype(str)
)
resolved = resolved.sort_values('q_date').reset_index(drop=True)
resolved['developer_prior_projects'] = (
    resolved.groupby('developer_clean').cumcount()
)


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 11 — STRUCTURAL SHIFT FLAGS
# post_order_2023: FERC Order 2023 (July 2023) restructured the queue process.
# Projects filed after this face materially different study rules.
# era: coarse pre/post 2019 flag capturing the solar+storage surge inflection.
# ═══════════════════════════════════════════════════════════════════════════════

resolved['post_order_2023'] = (
    resolved['q_date'] >= pd.Timestamp('2023-07-01')
).astype(int)

resolved['era'] = np.where(resolved['q_year'] < 2019, 'Pre-2019', '2019+')


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 12 — FINAL FEATURE LIST AND NULL CHECK
# ═══════════════════════════════════════════════════════════════════════════════

features = [
    # timing and process
    'q_year', 'q_quarter', 'days_in_queue', 'days_in_queue_imputed',
    'prop_lead_days', 'prop_date_missing',
    'ia_stage', 'has_ia', 'post_order_2023',
    # project characteristics
    'mw1', 'mw_total', 'is_clean', 'is_fossil', 'is_collocated', 'has_storage',
    # congestion — region level
    'region_year_congestion', 'congestion_yoy',
    # congestion — node level
    'poi_total_projects', 'poi_year_projects', 'poi_missing',
    # cluster dynamics
    'in_cluster', 'cluster_size',
    # developer behavior
    'developer_prior_projects',
]

null_check = pd.DataFrame({
    'dtype':    resolved[features].dtypes,
    'nulls':    resolved[features].isna().sum(),
    'null_pct': (resolved[features].isna().mean() * 100).round(1),
}).sort_values('nulls', ascending=False)

print("\n── Feature null check ───────────────────────────────────────")
print(null_check)
print(f"\nTotal rows:             {len(resolved):,}")
print(f"Rows with any null:     {resolved[features].isna().any(axis=1).sum():,}")
print(f"Withdrawn:              {resolved['withdrawn'].sum():,}  ({resolved['withdrawn'].mean():.1%})")
print(f"Operational/active:     {(resolved['withdrawn']==0).sum():,}")


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 13 — SPLIT DATASETS FOR DOWNSTREAM USE
# resolved_clf:  excludes suspended (ambiguous outcome for binary classifier)
# resolved_surv: keeps all, suspended treated as censored (event = 0)
# ═══════════════════════════════════════════════════════════════════════════════

resolved_clf  = resolved[resolved['q_status'] != 'suspended'].copy()
resolved_surv = resolved.copy()
resolved_surv['event'] = (resolved_surv['q_status'] == 'withdrawn').astype(int)

print(f"\n── Split datasets ───────────────────────────────────────────")
print(f"Classifier dataset:  {len(resolved_clf):,} rows  "
      f"({resolved_clf['withdrawn'].mean():.1%} withdrawn)")
print(f"Survival dataset:    {len(resolved_surv):,} rows  "
      f"({resolved_surv['event'].mean():.1%} events)")

Rows after initial drop: 36,436
Rows after dropping null q_date: 35,830
q_year range: 1970 – 2024

Outcome distribution:
q_status
withdrawn      20720
active         10466
operational     4188
suspended        456
Name: count, dtype: int64

Withdrawn: 20,720  (57.8%)

Withdrawn rows needing duration imputation: 10,748
Remaining nulls in days_in_queue: 0

IA stage distribution:
ia_stage
0    11952
1     5410
2     3664
3     5001
4     3091
5      132
6     6580
Name: count, dtype: int64

Technology flags:
  Collocated:  4,210  (11.7%)
  Has storage: 9,569  (26.7%)
  Clean:       30,228  (84.4%)
  Fossil:      4,096  (11.4%)

── Feature null check ───────────────────────────────────────
                            dtype  nulls  null_pct
q_year                    float64      0       0.0
is_fossil                   int64      0       0.0
cluster_size              float64      0       0.0
in_cluster                  int64      0       0.0
poi_missing                 int64      0       0.0

In [ ]:

# stop tracker and get emissions
import csv
emissions = tracker.stop()

log_file = "../outputs/codecarbon_emissions.csv"

print(f"Carbon cost this run: {emissions:.6f} kg CO2e")
print(f"Logged to: {log_file}")


Carbon cost this run: 0.000801 kg CO2e
Logged to: ../outputs/codecarbon_emissions.csv
